## Extracción de datos del IPC desde la API del INE

### Objetivo
Este notebook tiene como finalidad la extracción, exploración y estructuración de los datos del **Índice de Precios de Consumo (IPC)** publicados por el **Instituto Nacional de Estadística (INE)** a través de su API pública TEMPUS. 

A partir de la serie de datos IPC con periodicidad mensual, se construye un **modelo dimensional** compuesto por:
- **Dimensiones**: tabla de tiempo (*tiempo*), tabla de territorios (*territorio*), tabla de sectores IPC (*sectores_ipc*) y tabla de tipos de medida (*tipo_medida*).
- **Tabla de hechos**: tabla central (*ipc*) que relaciona cada observación del índice con sus dimensiones asociadas.

### Metodología
1. **Conexión a la API del INE** — Llamada al endpoint `/DATOS_TABLA/76136` para obtener los últimos 60 periodos mensuales.
2. **Extracción de dimensiones** — Recorrido de la respuesta JSON para poblar las tablas dimensionales, garantizando unicidad mediante filtros de clave.
3. **Construcción de la tabla de hechos** — Cruce de las dimensiones a través de identificadores primarios y asignación del valor del IPC a cada combinación (territorio, sector, medida, periodo).
4. **Exportación** — Volcado de cada tabla a ficheros CSV en `../files/data_raw/` para su consumo en etapas posteriores del pipeline.

### Contexto del proyecto
Estos datos se integran en un análisis más amplio sobre la **resiliencia empresarial en España**, donde se combinarán con información de constitución y disolución de empresas para estudiar la correlación entre el entorno macroeconómico (inflación) y la actividad empresarial.

In [1]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))

# Importación del módulo de conexión a la API
from src.api import connection_api
from src.api.config import API_URLS


C:\Users\fabih\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# Paso 1: Conexión y exploración inicial de la API

In [2]:
url = API_URLS["ipc"]

In [3]:
data = connection_api.llamada_api(url)

In [4]:
len(data)

1120

In [5]:
data[70]['Nombre']    #Probamos una posición random para ver si la información es correcta

'Andalucía. Vestido y calzado. Variación anual. '

In [6]:
data[0]['Data'][0]['Periodo']['Mes_inicio']     #Ruta al mes

'5'

In [7]:
data[0]['Data'][0]['Anyo']       #Ruta al año

2026

In [8]:
data[0]['Data'][0]['Periodo']['Nombre_largo']       #Ruta al nombre del mes

'Mayo'

In [9]:
data[0]['Data'][0]['Valor']             #Ruta al IPC

102.951

# Paso 2: Construcción de la dimensión tiempo

In [10]:
tiempo = {'id_tiempo': [], 'anio': [], 'mes': [], 'nombre_mes': [] }

for serie in data:
    for dato in serie['Data']:
        if dato['CodigoPeriodo'] not in tiempo['id_tiempo']:
            tiempo['id_tiempo'].append(dato['CodigoPeriodo'])
            tiempo['anio'].append(dato['Anyo'])
            tiempo['mes'].append(dato['Periodo']['Mes_inicio'])
            tiempo['nombre_mes'].append(dato['Periodo']['Nombre_largo'])


In [11]:
tiempo = pd.DataFrame(tiempo)
tiempo.sample()

,id_tiempo,anio,mes,nombre_mes
117,201608,2016,8,Agosto


In [12]:
tiempo.shape

(294, 4)

In [13]:
tiempo.to_csv('../files/data_raw/tiempo.csv', index=False)

# Paso 3: Construcción de la dimensión territorio

In [14]:
territorio = {'id_territorio': [], 'nombre_territorio': []}
id_ter = 1

for serie in data:
    nombre_completo = serie['Nombre']
    nombre_limpio = nombre_completo.split('.')[0].strip()

    if nombre_limpio not in territorio['nombre_territorio']:
        territorio['id_territorio'].append(id_ter)
        territorio['nombre_territorio'].append(nombre_limpio)

        id_ter += 1

In [15]:
territorio = pd.DataFrame(territorio)
territorio.sample()

,id_territorio,nombre_territorio
10,11,Comunitat Valenciana


In [16]:
territorio.shape

(20, 2)

In [17]:
territorio.to_csv('../files/data_raw/territorio.csv', index=False)

# Paso 4: Construcción de la dimensión sectores IPC

In [18]:
sectores_ipc = {'id_sector': [], 'nombre_sector': []}
id_sec = 1

for serie in data:
    nombre_completo = serie['Nombre']
    sector_limpio = nombre_completo.split('.')[1].strip()

    if sector_limpio not in sectores_ipc['nombre_sector']:
        sectores_ipc['id_sector'].append(id_sec)
        sectores_ipc['nombre_sector'].append(sector_limpio)

        id_sec += 1        

In [19]:
sectores_ipc = pd.DataFrame(sectores_ipc)

In [20]:
sectores_ipc.sample(10)

,id_sector,nombre_sector
2,3,Bebidas alcohólicas y tabaco
13,14,"Cuidado personal, protección social, y bienes ..."
8,9,Información y comunicaciones
12,13,Seguros y servicios financieros
9,10,"Actividades recreativas, deporte y cultura"
5,6,"Muebles, artículos del hogar y artículos para ..."
0,1,Índice general
6,7,Sanidad
4,5,"Vivienda, agua, electricidad, gas y otros comb..."
7,8,Transporte


In [21]:
sectores_ipc.to_csv('../files/data_raw/sectores_ipc.csv', index=False)

# Paso 5: Construcción de la dimensión tipo de medida

In [22]:
tipo_medida = {'id_medida': [], 'nombre_medida': []}
id_med = 1

for serie in data:
    nombre_completo = serie['Nombre']
    medida_limpio = nombre_completo.split('.')[2].strip()

    if medida_limpio not in tipo_medida['nombre_medida']:
        tipo_medida['id_medida'].append(id_med)
        tipo_medida['nombre_medida'].append(medida_limpio)

        id_med += 1     

In [23]:
tipo_medida = pd.DataFrame(tipo_medida)

In [24]:
tipo_medida.head()

,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [25]:
tipo_medida.to_csv('../files/data_raw/tipo_medida.csv', index=False)

# Paso 6: Construcción de la tabla de hechos IPC

In [26]:
# Creamos diccionarios

map_territorios = dict(zip(territorio['nombre_territorio'], territorio['id_territorio']))
map_sectores = dict(zip(sectores_ipc['nombre_sector'], sectores_ipc['id_sector']))
map_medidas = dict(zip(tipo_medida['nombre_medida'], tipo_medida['id_medida']))
 
ipc = {
    'id_tiempo': [],
    'id_territorio': [],
    'id_sector': [],
    'id_medida': [],
    'valor_ipc': []
}
 
# Extracción y Cruce de IDs
for serie in data:
    nombre_completo = serie['Nombre']
    partes = nombre_completo.split('.')
    if len(partes) >= 3:

        # Extraemos los nombres de texto limpio
        territorio_texto = partes[0].strip()
        sector_texto = partes[1].strip()
        medida_texto = partes[2].strip()

        # Miramos en nuestros mapas qué ID le toca a cada texto
        id_terr = map_territorios.get(territorio_texto)
        id_sec = map_sectores.get(sector_texto)
        id_med = map_medidas.get(medida_texto)

        # Si por algún motivo el INE da un texto que no guardamos antes, nos lo saltamos
        if id_terr is None or id_sec is None or id_med is None:
            continue

        # Bajamos al segundo nivel: los datos históricos de esta serie
        for dato in serie['Data']:
            valor = dato.get('Valor')
            id_time = dato.get('CodigoPeriodo')     # Sacamos el código del periodo para usarlo como ID de tiempo

            # Si tenemos todos los datos, appendeamos en la tabla
            if valor is not None and id_time is not None:
                ipc['id_tiempo'].append(str(id_time))
                ipc['id_territorio'].append(id_terr)
                ipc['id_sector'].append(id_sec)
                ipc['id_medida'].append(id_med)
                ipc['valor_ipc'].append(valor)
 
 

# Paso 7: Verificación y exportación de la tabla de hechos

In [27]:
ipc = pd.DataFrame(ipc)

ipc.sample(5)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
55289,200906,4,6,1,84.781
105486,202601,7,7,1,102.527
136311,202101,9,5,2,5.100
246714,202511,16,1,3,3.000
88543,202110,6,6,3,0.500


In [28]:
ipc.to_csv('../files/data_raw/ipc.csv', index=False)